In [1]:
!pip install pandas matplotlib numpy requests python-dotenv tidalapi

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
from dotenv import load_dotenv
from spotify_api import get_access_token as get_spotify_access_token
from tidal_api import create_session as create_tidal_session

load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
SPOTIFY_CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')
if not SPOTIFY_CLIENT_ID or not SPOTIFY_CLIENT_SECRET:
    raise ValueError("Spotify client ID and secret must be set in environment variables.")

SPOT_HEADERS = get_spotify_access_token(SPOTIFY_CLIENT_ID, SPOTIFY_CLIENT_SECRET)
print("Authenticated with Spotify successfully.")

TIDAL_SESSION = create_tidal_session(Path("tidal_session.json"))
print("Authenticated with TIDAL successfully.")

/home/shearer/personal/playlist_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authenticated with Spotify successfully.
Authenticated with TIDAL successfully.


In [3]:
from spotify_api import get_artist_song_ids
from tidal_api import spotify_to_tidal_ids
ARTIST_NAMES: list[str] = ["Drake"]
spotify_song_ids = get_artist_song_ids(ARTIST_NAMES[0], SPOT_HEADERS)
print(spotify_song_ids[:10])
tidal_song_ids = spotify_to_tidal_ids(spotify_song_ids[:25], SPOT_HEADERS, TIDAL_SESSION)
print(tidal_song_ids[:10])

Found artist: Drake (ID: 3TVXtAsR1Inumwj472S9r4)
Found 82 albums/singles/compilations
Found 405 unique tracks
['27Tqjd2nMW2mIp6Yk1ovwY', '6hHKFReuQR9VQp39ev43wY', '4BhGTc3Cgay2U1QcTS7vQe', '0GR7iJLhj80KD5LkA14ZRn', '0JaVdpmiex2EP7bBzyKVTa', '4PA16FAl8LPmFmOhARawdV', '4AtXMDQiztBcGVcLTBrcCQ', '4alHkxxwAhvoGg3dJCATKV', '6N7CsPNdryWPtG5CTmCHpx', '6G8kHiVZ1jW7vHMPVRNZU0']


Looking up TIDAL IDs: 100%|██████████| 25/25 [00:02<00:00, 12.40it/s]

[{'spotify_id': '27Tqjd2nMW2mIp6Yk1ovwY', 'isrc': 'USUG11902756', 'tidal_id': '114585794'}, {'spotify_id': '6hHKFReuQR9VQp39ev43wY', 'isrc': 'USCM51900151', 'tidal_id': '104096995'}, {'spotify_id': '4BhGTc3Cgay2U1QcTS7vQe', 'isrc': 'USCM51600092', 'tidal_id': '60100936'}, {'spotify_id': '0GR7iJLhj80KD5LkA14ZRn', 'isrc': 'USCM51500292', 'tidal_id': '51579773'}, {'spotify_id': '0JaVdpmiex2EP7bBzyKVTa', 'isrc': 'USCM51900155', 'tidal_id': '104096984'}, {'spotify_id': '4PA16FAl8LPmFmOhARawdV', 'isrc': 'USCM51600070', 'tidal_id': '77690156'}, {'spotify_id': '4AtXMDQiztBcGVcLTBrcCQ', 'isrc': 'USWB12000196', 'tidal_id': '288088379'}, {'spotify_id': '4alHkxxwAhvoGg3dJCATKV', 'isrc': 'USUG11902752', 'tidal_id': '114585790'}, {'spotify_id': '6N7CsPNdryWPtG5CTmCHpx', 'isrc': 'USUG12306086', 'tidal_id': '320189500'}, {'spotify_id': '6G8kHiVZ1jW7vHMPVRNZU0', 'isrc': 'USCM51800208', 'tidal_id': '92761305'}]


In [4]:
# ── Create / ensure DB tables ─────────────────────────────────────────
from db_connection import db, Songs, SongAudioFeatures

db.connect(reuse_if_open=True)
db.create_tables([Songs, SongAudioFeatures], safe=True)
print("Tables ready.")
db.close()

Tables ready.


True

In [5]:
# ── Filter out songs already in the DB ────────────────────────────────
db.connect(reuse_if_open=True)
existing_spotify_ids = set(s.spotify_id for s in Songs.select(Songs.spotify_id))
db.close()

before = len(spotify_song_ids)
spotify_song_ids = [sid for sid in spotify_song_ids if sid not in existing_spotify_ids]
print(f"Filtered {before - len(spotify_song_ids)} existing songs, {len(spotify_song_ids)} remaining")

# Re-derive TIDAL IDs for the filtered list
tidal_song_ids = spotify_to_tidal_ids(spotify_song_ids, SPOT_HEADERS, TIDAL_SESSION)
print(f"Mapped {len(tidal_song_ids)} songs to TIDAL IDs")

Filtered 359 existing songs, 46 remaining


Looking up TIDAL IDs: 100%|██████████| 46/46 [00:03<00:00, 12.52it/s]

Mapped 46 songs to TIDAL IDs


In [6]:
# ── Fetch full Spotify metadata & insert Songs into DB ────────────────
import uuid
from spotify_api import get_tracks_metadata

tracks_meta = get_tracks_metadata(spotify_song_ids, SPOT_HEADERS)
print(f"Fetched metadata for {len(tracks_meta)} tracks")

# Build a lookup from spotify_id -> tidal_id
tidal_lookup = {entry["spotify_id"]: entry["tidal_id"] for entry in tidal_song_ids}

db.connect(reuse_if_open=True)
inserted, skipped = 0, 0
for meta in tracks_meta:
    try:
        Songs.create(
            id=uuid.uuid4(),
            spotify_id=meta["id"],
            tidal_id=tidal_lookup.get(meta["id"]),
            song_name=meta["name"],
            artists=meta["artists"],
            album_name=meta.get("album_name"),
            album_total_tracks=meta.get("album_total_tracks"),
            release_date=meta.get("album_release_date"),
            disc_number=meta.get("disc_number"),
            duration_ms=meta.get("duration_ms"),
            explicit=meta.get("explicit"),
            isrc=meta.get("isrc"),
            spotify_external_url=meta.get("external_url"),
            spotify_href=meta.get("href"),
            popularity=meta.get("popularity"),
            track_number=meta.get("track_number"),
            spotify_uri=meta.get("uri"),
        )
        inserted += 1
    except Exception as e:
        skipped += 1  # likely duplicate song_name or spotify_id

db.close()
print(f"Inserted {inserted} songs, skipped {skipped} duplicates")

Fetched metadata for 46 tracks
Inserted 0 songs, skipped 46 duplicates


In [7]:
# ── Download songs from TIDAL ─────────────────────────────────────────
from tidal_api import download_songs

downloaded_paths = download_songs(tidal_song_ids)
print(f"Downloaded {len(downloaded_paths)} tracks")

Downloaded 41 tracks


In [8]:
# ── Extract audio features & store in SongAudioFeatures ───────────────
from pathlib import Path
from audio_processor import process_bulk

# Gather downloaded audio files keyed by tidal_id
downloads_dir = Path("OrpheusDL/downloads")
audio_files = {}
for entry in tidal_song_ids:
    tid = entry["tidal_id"]
    if tid is None:
        continue
    path = downloads_dir / f"{tid}.m4a"
    if path.exists():
        audio_files[tid] = path

print(f"Found {len(audio_files)} audio files to analyse")

# Bulk analyse
paths_list = list(audio_files.values())
tidal_ids_list = list(audio_files.keys())
analysis_results = process_bulk(paths_list)

# Store in DB
db.connect(reuse_if_open=True)
stored, failed = 0, 0
for tid, analysis in zip(tidal_ids_list, analysis_results):
    if "error" in analysis:
        print(f"[SKIP] {tid}: {analysis['error']}")
        failed += 1
        continue

    song = Songs.get_or_none(Songs.tidal_id == tid)
    if song is None:
        failed += 1
        continue

    track_info = analysis.get("track", {})
    try:
        SongAudioFeatures.create(
            song=song.id,
            acousticness=analysis.get("acousticness"),
            danceability=analysis.get("danceability"),
            energy=analysis.get("energy"),
            instrumentalness=analysis.get("instrumentalness"),
            key=analysis.get("key"),
            liveness=analysis.get("liveness"),
            loudness=analysis.get("loudness"),
            mode=analysis.get("mode"),
            speechiness=analysis.get("speechiness"),
            tempo=analysis.get("tempo"),
            time_signature=analysis.get("time_signature"),
            valence=analysis.get("valence"),
            num_samples=track_info.get("num_samples"),
            duration=track_info.get("duration"),
            sample_md5=track_info.get("sample_md5"),
            end_of_fade_in=track_info.get("end_of_fade_in"),
            start_of_fade_out=track_info.get("start_of_fade_out"),
            tempo_confidence=track_info.get("tempo_confidence"),
            time_signature_confidence=track_info.get("time_signature_confidence"),
            key_confidence=track_info.get("key_confidence"),
            mode_confidence=track_info.get("mode_confidence"),
            bars=analysis.get("bars"),
            beats=analysis.get("beats"),
            sections=analysis.get("sections"),
            segments=analysis.get("segments"),
            tatums=analysis.get("tatums"),
        )
        stored += 1
    except Exception as e:
        print(f"[SKIP] {tid}: {e}")
        failed += 1

db.close()
print(f"Stored audio features for {stored} tracks, failed {failed}")

Found 40 audio files to analyse


Processing tracks: 100%|██████████| 40/40 [00:28<00:00,  1.43track/s]


[SKIP] 36313417: duplicate key value violates unique constraint "songaudiofeatures_song_id"
DETAIL:  Key (song_id)=(03e2fe95-1dc3-4e30-ab0b-6f8edb6da747) already exists.

[SKIP] 329293985: duplicate key value violates unique constraint "songaudiofeatures_song_id"
DETAIL:  Key (song_id)=(07d784d8-e351-41f8-9e0a-4dc94b386755) already exists.

[SKIP] 329297591: duplicate key value violates unique constraint "songaudiofeatures_song_id"
DETAIL:  Key (song_id)=(d018d235-6bf0-4063-abbb-85e866d92562) already exists.

[SKIP] 22563751: duplicate key value violates unique constraint "songaudiofeatures_song_id"
DETAIL:  Key (song_id)=(1a5a5e72-9030-4946-88a1-3150529e00de) already exists.

[SKIP] 329294014: duplicate key value violates unique constraint "songaudiofeatures_song_id"
DETAIL:  Key (song_id)=(4e8a8705-d4c6-4287-81f9-705ffae729d5) already exists.

[SKIP] 3559471: duplicate key value violates unique constraint "songaudiofeatures_song_id"
DETAIL:  Key (song_id)=(db1d4923-eb8d-44c0-a612-ad7

In [9]:
# ── Fetch lyrics from TIDAL & store in DB ─────────────────────────────
from tidal_api import get_track_lyrics
from tqdm.auto import tqdm

db.connect(reuse_if_open=True)
songs_needing_lyrics = (
    Songs.select()
    .where(Songs.tidal_id.is_null(False) & (Songs.lyrics.is_null(True) | (Songs.lyrics == "")))
)
songs_list = list(songs_needing_lyrics)
print(f"Songs needing lyrics: {len(songs_list)}")

fetched, missed = 0, 0
for song in tqdm(songs_list, desc="Fetching lyrics"):
    lyrics = get_track_lyrics(song.tidal_id, TIDAL_SESSION)
    if lyrics:
        Songs.update(lyrics=lyrics).where(Songs.id == song.id).execute()
        fetched += 1
    else:
        missed += 1

db.close()
print(f"Fetched lyrics for {fetched} songs, {missed} not available")

Songs needing lyrics: 23


Fetching lyrics:   0%|          | 0/23 [00:00<?, ?it/s]Track '4005079' is unavailable
Track '35032342' is unavailable
Fetching lyrics:   9%|▊         | 2/23 [00:00<00:01, 18.96it/s]Track '37061723' is unavailable
Track '5476063' is unavailable
Fetching lyrics:  17%|█▋        | 4/23 [00:00<00:00, 19.41it/s]Track '34940903' is unavailable
Track '88460362' is unavailable
Fetching lyrics:  26%|██▌       | 6/23 [00:00<00:00, 18.91it/s]Track '45771071' is unavailable
Track '45394163' is unavailable
Fetching lyrics:  35%|███▍      | 8/23 [00:00<00:00, 18.14it/s]Track '35452621' is unavailable
Track '36208123' is unavailable
Fetching lyrics:  43%|████▎     | 10/23 [00:00<00:00, 18.42it/s]Track '45771076' is unavailable
Track '45771081' is unavailable
Fetching lyrics:  52%|█████▏    | 12/23 [00:00<00:00, 18.58it/s]Track '45771079' is unavailable
Track '4005086' is unavailable
Fetching lyrics:  61%|██████    | 14/23 [00:00<00:00, 18.25it/s]Track '35452620' is unavailable
Track '45771078' is unav

Fetched lyrics for 0 songs, 23 not available


In [10]:
# ── Transcribe lyrics for songs still missing them ────────────────────
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

TRANSCRIPTION_PROMPT = """This is a song recording. Transcribe all lyrics exactly as heard, following these rules:
- Preserve all repetitions, including choruses sung multiple times
- If lyrics switch language (e.g. Spanish, French), transcribe them in their original language
- Distinguish spoken/ad-libbed lines with parentheses, e.g. (spoken: yeah, that's right)
- If a word or phrase is genuinely unclear, write [unclear] rather than guessing
- Use one blank line between song sections; do not add section labels
- Transcribe informal or slurred speech phonetically as heard, not corrected to standard spelling
- Include background vocals in brackets, e.g. [background: oh yeah]"""

db.connect(reuse_if_open=True)
still_missing = (
    Songs.select()
    .where(Songs.tidal_id.is_null(False) & (Songs.lyrics.is_null(True) | (Songs.lyrics == "")))
)
songs_to_transcribe = [
    {"id": str(s.id), "tidal_id": s.tidal_id}
    for s in still_missing
    if (downloads_dir / f"{s.tidal_id}.m4a").exists()
]
print(f"Songs to transcribe via audio: {len(songs_to_transcribe)}")


def _transcribe_one(song):
    audio_file = downloads_dir / f"{song['tidal_id']}.m4a"
    result = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
        response_format="text",
        prompt=TRANSCRIPTION_PROMPT,
    )
    lyrics = result.strip()
    Songs.update(lyrics=lyrics).where(Songs.id == song["id"]).execute()
    return {"id": song["id"], "tidal_id": song["tidal_id"], "lyrics": lyrics}


transcribed = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(_transcribe_one, s): s for s in songs_to_transcribe}
    for future in as_completed(futures):
        song = futures[future]
        try:
            out = future.result()
            transcribed.append(out)
        except Exception as e:
            print(f"[ERROR] {song['tidal_id']}: {e}")

db.close()
print(f"Transcribed lyrics for {len(transcribed)} songs")

Songs to transcribe via audio: 0
Transcribed lyrics for 0 songs


In [11]:
# ── Generate lyrics embeddings & store in DB ──────────────────────────
# Embed each line separately and mean-pool so full lyrics are captured
from embedding_model import create_text_embeddings
import torch

db.connect(reuse_if_open=True)
songs_with_lyrics = (
    Songs.select(Songs, SongAudioFeatures)
    .join(SongAudioFeatures, on=(SongAudioFeatures.song == Songs.id))
    .where(
        Songs.lyrics.is_null(False)
        & (Songs.lyrics != "")
        & SongAudioFeatures.lyrics_embeddings.is_null(True)
    )
)
songs_list = list(songs_with_lyrics)
print(f"Songs needing lyrics embeddings: {len(songs_list)}")

embedded = 0
for song in tqdm(songs_list, desc="Lyrics embeddings"):
    lines = [line.strip() for line in song.lyrics.splitlines() if line.strip()]
    if not lines:
        continue
    line_embeddings = create_text_embeddings(lines)           # (n_lines, 768)
    song_embedding = line_embeddings.mean(dim=0)              # (768,)
    song_embedding = song_embedding / song_embedding.norm()   # re-normalise
    (SongAudioFeatures
     .update(lyrics_embeddings=song_embedding.cpu().tolist())
     .where(SongAudioFeatures.song == song.id)
     .execute())
    embedded += 1

db.close()
print(f"Stored lyrics embeddings for {embedded} songs")

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 1663.40it/s, Materializing param=vision_model.post_layernorm.weight]                      


Songs needing lyrics embeddings: 246


Lyrics embeddings: 100%|██████████| 246/246 [02:54<00:00,  1.41it/s]

Stored lyrics embeddings for 246 songs


In [12]:
# ── Extract album art → image embeddings & store in DB ────────────────
from file_metadata_processor import extract_image_from_m4a_mutagen
from embedding_model import create_images_embeddings

db.connect(reuse_if_open=True)
songs_needing_images = (
    Songs.select(Songs, SongAudioFeatures)
    .join(SongAudioFeatures, on=(SongAudioFeatures.song == Songs.id))
    .where(
        Songs.tidal_id.is_null(False)
        & SongAudioFeatures.image_embeddings.is_null(True)
    )
)
songs_list = list(songs_needing_images)
print(f"Songs needing image embeddings: {len(songs_list)}")

BATCH_SIZE = 32
embedded = 0
batch_songs = []
batch_images = []

for song in tqdm(songs_list, desc="Extracting album art"):
    audio_path = downloads_dir / f"{song.tidal_id}.m4a"
    if not audio_path.exists():
        continue
    img = extract_image_from_m4a_mutagen(str(audio_path))
    if img is None:
        continue
    batch_songs.append(song)
    batch_images.append(img)

    if len(batch_images) >= BATCH_SIZE:
        embeddings = create_images_embeddings(batch_images)
        embeddings_list = embeddings.cpu().tolist()
        for s, emb in zip(batch_songs, embeddings_list):
            (SongAudioFeatures
             .update(image_embeddings=emb)
             .where(SongAudioFeatures.song == s.id)
             .execute())
            embedded += 1
        batch_songs, batch_images = [], []

# Process remaining batch
if batch_images:
    embeddings = create_images_embeddings(batch_images)
    embeddings_list = embeddings.cpu().tolist()
    for s, emb in zip(batch_songs, embeddings_list):
        (SongAudioFeatures
         .update(image_embeddings=emb)
         .where(SongAudioFeatures.song == s.id)
         .execute())
        embedded += 1

db.close()
print(f"Stored image embeddings for {embedded} songs")

Songs needing image embeddings: 0


Extracting album art: 0it [00:00, ?it/s]

Stored image embeddings for 0 songs
